# ODIL Warm-Start for PIFT — Novel Contribution

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cmhobbs96/pift-od-il-inverse-problems/blob/main/examples/04_odil_warmstart.ipynb)

This notebook investigates a **novel contribution** of this project: using the ODIL MAP
solution (notebook 03) as a warm-start for PIFT SGLD (notebook 01).

The key idea is that ODIL finds the high-probability mode of the posterior extremely fast
(sub-millisecond), while PIFT SGLD explores the full posterior but requires many steps to
reach that mode from a random or FD initialization. By initializing SGLD at the ODIL MAP,
we expect:
- **Shorter effective burn-in** — the chain starts near the posterior mode.
- **Faster convergence** of the running posterior-mean L2 error.
- **Equal asymptotic quality** — all warm-starts should converge to the same posterior.

We compare three initialization strategies in an **ablation study**:

| Mode | $\theta_0$ | Description |
|------|-----------|-------------|
| `cold` | $\mathbf{0}$ | Zero initialization (baseline) |
| `fd` | FD $\to$ lstsq projection | Finite-difference warm-start |
| `odil` | ODIL GN $\to$ basis projection | ODIL MAP warm-start (novel) |

In [ ]:
%pip install -q git+https://github.com/cmhobbs96/pift-od-il-inverse-problems.git
%pip install -q tqdm

import jax
jax.config.update('jax_enable_x64', True)
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

print('JAX backend:', jax.default_backend(), '| devices:', jax.devices())

from core.parameterizations import SineBasisField
from pipelines.common import forcing, phi_true
from pipelines.phase_a import run_phase_a_forward_poisson

def show_fig(fig, dpi=120):
    import tempfile
    from IPython.display import Image, display
    with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as f:
        fig.savefig(f.name, dpi=dpi, bbox_inches='tight')
        plt.close(fig)
        display(Image(f.name))

## Configuration

In [ ]:
CONFIG = {
    # --- Shared problem ---
    'seed':            7,       # reproducibility                               [0, 2**31-1]
    'n_obs':           28,      # noisy observations                            [4, 200]
    'noise_std':       0.08,    # observation noise sigma                       [0.0, 0.5]
    'n_truth_modes':   6,       # sine modes in random truth                    [1, 20]
    'truth_amp_decay': 1.5,     # amplitude decay of truth                      [0.0, 4.0]
    'n_modes':         12,      # PIFT sine-basis modes                         [4, 64]
    'n_quad':          96,      # quadrature points per SGLD step               [16, 512]
    'n_grid':          300,     # plotting / reference grid                     [50, 2000]
    'beta':            0.5,     # physics trust beta                            [0.01, 100]
    # --- PIFT sampling ---
    'n_steps':         8000,    # SGLD iterations per init mode                 [1000, 100000]
    'burn_in':         1500,    # warm-up discarded                             [100, n_steps/2]
    'thin':            8,       # thinning factor                               [1, 100]
    'step_size0':      2e-3,    # initial SGLD step                             [1e-5, 1e-2]
    'decay':           0.55,    # step-size decay exponent                      [0.5, 1.0]
    # --- ODIL warm-start ---
    'odil_n_grid':     257,     # FD grid for ODIL                              [32, 4096]
    'odil_max_iter':   50,      # max ODIL GN iterations                        [5, 500]
}

## Warm-Start Ablation

For each initialization mode we call `run_phase_a_forward_poisson` with `init_mode=mode`.
The pipeline returns the full chain and a `running_l2` array (running posterior-mean L2
computed every `checkpoint_every` steps), which we use to compare convergence speed.

In [ ]:
modes = ['cold', 'fd', 'odil']
ablation_results = {}

for mode in modes:
    print(f'\n--- Running init_mode = {mode!r} ---')
    res = run_phase_a_forward_poisson(
        config=CONFIG,
        init_mode=mode,
        save_outputs=False,
        checkpoint_every=200,
    )
    ablation_results[mode] = res
    print(
        f'  status={res["status"]}  '
        f'n_samples={res["n_samples"]}  '
        f'final_l2={res["l2_error"]:.4f}  '
        f'runtime={res["runtime_s"]:.1f}s'
    )

In [ ]:
# --- Helper: compute running posterior-mean L2 ---
def running_l2(chain, basis_grid, phi_truth_grid, min_samples=1):
    """L2 error of the running mean as more samples are included."""
    n = len(chain)
    l2s = []
    cumsum = np.zeros(basis_grid.shape[0])
    for i in range(n):
        cumsum += basis_grid @ chain[i]
        if i + 1 >= min_samples:
            mean_i = cumsum / (i + 1)
            l2s.append(float(np.sqrt(np.mean((mean_i - phi_truth_grid) ** 2))))
    return np.array(l2s)

# Grid and truth
x_grid = np.linspace(0, 1, CONFIG['n_grid'])
k_vec = np.arange(1, CONFIG['n_modes'] + 1)
basis_grid = np.sin(np.pi * k_vec[None, :] * x_grid[:, None])

# Truth on grid (use pipeline's phi_true)
phi_truth_grid = phi_true(x_grid)

# --- Build running L2 curves ---
colors = {'cold': 'tomato', 'fd': 'steelblue', 'odil': 'seagreen'}
styles = {'cold': '-',       'fd': '--',        'odil': '-.'}

fig, ax = plt.subplots(figsize=(9, 5))

for mode in modes:
    res = ablation_results[mode]
    # Use pre-computed running_l2 from pipeline if available
    if 'running_l2' in res and res['running_l2'] is not None:
        rl2 = np.array(res['running_l2'])
        steps = np.linspace(0, CONFIG['n_steps'], len(rl2))
    else:
        chain = np.array(res['chain_thinned'])
        rl2 = running_l2(chain, basis_grid, phi_truth_grid)
        steps = np.linspace(CONFIG['burn_in'], CONFIG['n_steps'], len(rl2))
    final_l2 = res['l2_error']
    ax.semilogy(
        steps, rl2,
        color=colors[mode], linestyle=styles[mode], lw=2,
        label=f'{mode} (final L2={final_l2:.3f})'
    )

# Target L2 reference line
ax.axhline(0.05, color='gray', linestyle=':', lw=1.5, label='Target L2 = 0.05')
ax.axvline(CONFIG['burn_in'], color='black', linestyle=':', lw=1, alpha=0.5, label='Burn-in end')

ax.set_xlabel('SGLD step')
ax.set_ylabel('Running posterior-mean L2 error')
ax.set_title('Warm-Start Ablation: Cold vs FD vs ODIL Initialization')
ax.legend(fontsize=10)
ax.set_xlim(left=0)
fig.tight_layout()
show_fig(fig)

## Interpretation

**Expected findings:**

- **Cold start** begins with high L2 (the zero initialization is far from the posterior mode)
  and converges slowly — the SGLD chain must traverse a long path before mixing.
- **FD warm-start** starts closer to the truth; the running L2 drops faster in the early
  steps, reaching the target L2 threshold well before the cold chain.
- **ODIL warm-start** achieves comparable or better early-step L2 to the FD warm-start,
  despite using an entirely different solver (discrete FD residual vs. continuous energy).
  Crucially, the ODIL solve takes $<1$ ms vs. the FD lstsq projection.
- **Asymptotic convergence:** all three modes converge to the same posterior distribution;
  the warm-start only affects *how quickly* the chain reaches stationarity.

This demonstrates that ODIL can serve as a lightweight, scalable warm-start mechanism for
PIFT SGLD without requiring access to a separate FD solver infrastructure.